<a href="https://colab.research.google.com/github/niall-calvert/KDA-Decode-Kernel/blob/main/KDA_decode_kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax.numpy as jnp


def matmul(
    x: jax.Array,
    y: jax.Array,
) -> jax.Array:
    """Multiply two matrices using naive JAX.

    Args:
        x: Left input matrix with shape (M, K).
        y: Right input matrix with shape (K, N).

    Returns:
        The matrix product with shape (M, N).
    """
    return x @ y

In [ ]:
import jax
import jax.numpy as jnp


def acc_dtype(input_dtype) -> jnp.dtype:
    """Accumulator dtype: fp64 for fp64 inputs, fp32 otherwise."""
    return jnp.float64 if input_dtype == jnp.float64 else jnp.float32


def naive_recurrent_kda(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    g: jax.Array,
    beta: jax.Array,
    scale: float | None = None,
    initial_state: jax.Array | None = None,
    output_final_state: bool = False,
) -> tuple[jax.Array, jax.Array | None]:
    """
    Core recurrence (per timestep):
        S' = S_{t-1} * exp(g_t)                            decay
        residual = v_t - k_t^T @ S'                        prediction error
        S_t = S' + beta_t * k_t ⊗ residual                 delta update
        o_t = (q_t * scale)^T @ S_t                        output

    Dtype behavior (matching FLA):
      - All inputs cast to fp32 for computation
      - Hidden state S is fp32 accumulator
      - Output o computed in fp32, cast back to original dtype
      - Final state S stays in fp32
      - fp64 mode: all computation in fp64, no precision cast

    Args:
        q:               [B, T, H, K] — Queries
        k:               [B, T, H, K] — Keys
        v:               [B, T, H, V] — Values
        g:               [B, T, H, K] — Per-element gate in log-space (e.g., -exp(A)*softplus(g))
        beta:            [B, T, H]    — Learning rate / step size for delta rule
        scale:           Scalar query scale. Defaults to K ** -0.5.
        initial_state:   [B, H, K, V] — Initial hidden state. Optional.
        output_final_state: Whether to return the final hidden state.

    Returns:
        o:           [B, T, H, V] — Output (original input dtype)
        final_state: [B, H, K, V] in fp32 (or fp64), or None
    """
    orig_dtype, acc_dt = v.dtype, acc_dtype(q.dtype)

    assert q.ndim == 4, f"q must be 4D [B,T,H,K], got {q.ndim}D"
    assert k.shape == q.shape, f"k shape {k.shape} != q shape {q.shape}"
    assert (
        v.ndim == 4 and v.shape[:3] == q.shape[:3]
    ), f"v shape {v.shape} incompatible with q shape {q.shape}"
    assert g.ndim == 4 and g.shape == q.shape, f"g shape {g.shape} != q shape {q.shape}"
    assert beta.ndim == 3 and beta.shape == q.shape[:3], f"beta shape {beta.shape} != {q.shape[:3]}"

    B, T, H, K, V = *q.shape, v.shape[-1]

    if initial_state is not None:
        assert initial_state.shape == (
            B,
            H,
            K,
            V,
        ), f"initial_state shape {initial_state.shape} != ({B}, {H}, {K}, {V})"

    if scale is None:
        scale = K**-0.5

    # [B, T, H, K] -> [B, H, T, K], cast to acc_dt
    q, k, v, g = (jnp.transpose(x, (0, 2, 1, 3)).astype(acc_dt) for x in (q, k, v, g))
    # q: [B, H, T, K]   k: [B, H, T, K]   v: [B, H, T, V]   g: [B, H, T, K]

    # [B, T, H] -> [B, H, T]
    beta = jnp.transpose(beta, (0, 2, 1)).astype(acc_dt)  # [B, H, T]

    q = q * scale  # [B, H, T, K]

    S = jnp.zeros((B, H, K, V), dtype=acc_dt)  # [B, H, K, V] hidden state
    if initial_state is not None:
        S += initial_state.astype(acc_dt)  # [B, H, K, V]
    o = jnp.zeros((B, H, T, V), dtype=acc_dt)  # [B, H, T, V] output buffer

    for i in range(T):
        q_i = q[:, :, i]  # [B, H, K]
        k_i = k[:, :, i]  # [B, H, K]
        v_i = v[:, :, i]  # [B, H, V]
        g_i = g[:, :, i]  # [B, H, K]
        b_i = beta[:, :, i]  # [B, H]

        # 1. Decay the state
        # exp(g_i): [B, H, K] -> [B, H, K, 1] via broadcast
        S = S * jnp.exp(g_i)[..., None]  # [B, H, K, V]

        # 2. Delta rule update
        # k_i[..., None]: [B, H, K, 1],  k_i[..., None] * S: [B, H, K, V]
        v_predicted = (k_i[..., None] * S).sum(-2)  # [B, H, V]
        residual = v_i - v_predicted  # [B, H, V]

        # b_i[..., None] * k_i: [B, H, K],  einsum -> [B, H, K, V]
        S = S + jnp.einsum("bhk,bhv->bhkv", b_i[..., None] * k_i, residual)  # [B, H, K, V]

        # 3. Compute output: einsum [B,H,K] x [B,H,K,V] -> [B, H, V]
        o = o.at[:, :, i].set(jnp.einsum("bhk,bhkv->bhv", q_i, S))  # [B, H, V]

    final_state = S if output_final_state else None  # [B, H, K, V] or None
    # [B, H, T, V] -> [B, T, H, V], cast back to orig_dtype
    return jnp.transpose(o, (0, 2, 1, 3)).astype(orig_dtype), final_state

In [ ]:
from functools import partial

import jax
from jax.experimental import pallas as pl
import jax.numpy as jnp
import numpy as np

def matmul_kernel(q_ref,
                  k_ref,
                  v_ref,
                  g_ref,
                  beta_ref,
                  scale_ref,
                  init_ref,
                  out_ref):
  q = q_ref[...]
  v = v_ref[...]
  B, T, H, K, V = *q.shape, v.shape[-1]
  orig_dtype, acc_dt = v.dtype, acc_dtype(q.dtype)
  k = k_ref[...].astype(acc_dt)
  g = g_ref[...].astype(acc_dt)
  beta = beta_ref[...].astype(acc_dt)
  scale = scale_ref[...]
  init = init_ref[...]
  out = out_ref[...]


  q = jnp.squeeze(q, axis=1)  # [B, H, K]
  k = jnp.squeeze(k, axis=1)  # [B, H, K]
  v = jnp.squeeze(v, axis=1)  # [B, H, V]
  g = jnp.squeeze(g, axis=1)  # [B, H, K]
  b = jnp.squeeze(beta, axis=1)  # [B, H]

  if scale is None:
        scale = K**-0.5

  q = q * scale  # [B, H, K]


  ## anything in here will be done on chip. this can be anything done for the attention operations
  S = jnp.zeros((B, H, K, V), dtype=acc_dt)  # [B, H, K, V] hidden state
  if init is not None:
      S += init.astype(acc_dt)  # [B, H, K, V]
  o = jnp.zeros((B, H, T, V), dtype=acc_dt)  # [B, H, T, V] output buffer

  # 1. Decay the state
  # exp(g_i): [B, H, K] -> [B, H, K, 1] via broadcast
  S = S * jnp.exp(g)[..., None]  # [B, H, K, V]

  # 2. Delta rule update
  # k_i[..., None]: [B, H, K, 1],  k_i[..., None] * S: [B, H, K, V]
  v_predicted = (k[..., None] * S).sum(-2)  # [B, H, V]
  residual = v - v_predicted  # [B, H, V]

  # b_i[..., None] * k_i: [B, H, K],  einsum -> [B, H, K, V]
  S = S + jnp.einsum("bhk,bhv->bhkv", b[..., None] * k, residual)  # [B, H, K, V]

  # 3. Compute output: einsum [B,H,K] x [B,H,K,V] -> [B, H, V]
  o = o.at[:, :].set(jnp.einsum("bhk,bhkv->bhv", q, S))  # [B, H, V]

  final_state = S if out else None  # [B, H, K, V] or None
  # [B, H, T, V] -> [B, T, H, V], cast back to orig_dtype
  return jnp.transpose(o, (0, 2, 1, 3)).astype(orig_dtype), final_state



def naive_recurrent_kda(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    g: jax.Array,
    beta: jax.Array,
    scale: float | None = None,
    initial_state: jax.Array | None = None,
    output_final_state: bool = False,
) -> tuple[jax.Array, jax.Array | None]:

  orig_dtype, acc_dt = v.dtype, acc_dtype(q.dtype)

  B, T, H, K, V = *q.shape, v.shape[-1]

  if scale is None:
      scale = K**-0.5

  if initial_state is None:
    initial_state = jnp.zeros(
        (B, H, K, V),
        dtype=acc_dt,
    )


  BM = 128
  BN = 128
  BK = 128

  return pl.pallas_call(
    matmul_kernel,
    out_shape=jax.ShapeDtypeStruct((x.shape[0], y.shape[1]), x.dtype),

    grid = (
      M // BM,
      N // BN,
      K // BK,
    ),

    in_specs=[
        pl.BlockSpec((x.shape[0] // 2, x.shape[1]), lambda i, j: (i, 0)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j)),
        pl.BlockSpec((y.shape[0], y.shape[1] // 2), lambda i, j: (0, j))
    ],
    out_specs=pl.BlockSpec(
        (x.shape[0] // 2, y.shape[1] // 2), lambda i, j: (i, j),
    )
  )(q, k, v, g, beta, scale, initial_state, output_final_state)